In [52]:
import pandas as pd
import numpy as np
import datetime as dt

session_df = pd.read_csv("../data/analysis_table/session_table.csv")
user_df = pd.read_csv("../data/analysis_table/user_table.csv")
funnel = pd.read_csv("../data/analysis_table/funnel_table.csv")

In [53]:
# 1. Conversion Rate
view_sessions = funnel.loc[funnel['stage'] == 'view', 'sessions'].iloc[0]
funnel['conversion_rate'] = funnel['sessions'] / view_sessions

print(funnel)

   Unnamed: 0     stage  sessions  conversion_rate
0           0      view     34368         1.000000
1           1      cart      1243         0.036167
2           2  purchase      2133         0.062064


In [54]:
# 2. Drop-off Rate
# calculate conversion rate of each steps
funnel['step_conversion_rate'] = (funnel['sessions'] / funnel['sessions'].shift(1))

# set the first step(NaN) to 1
funnel.loc[0, 'step_conversion_rate'] = 1 

# calculate drop-off rate
funnel['drop_off_date'] = 1 - funnel['step_conversion_rate']

print(funnel)

   Unnamed: 0     stage  sessions  conversion_rate  step_conversion_rate  \
0           0      view     34368         1.000000              1.000000   
1           1      cart      1243         0.036167              0.036167   
2           2  purchase      2133         0.062064              1.716010   

   drop_off_date  
0       0.000000  
1       0.963833  
2      -0.716010  


In [55]:
# 3. Time to Convert: how much time does viewing to purchasing takes
oct_2019 = pd.read_csv("../data/analysis_table/2019_oct_cleaned.csv")

view_times = (
    oct_2019[oct_2019['event_type'] == 'view'].groupby(['user_id', 'user_session'])['event_time']
    .min()   # take the min as first view time
    .reset_index()
    .rename(columns = {'event_time' : "first_view_time"})
)

view_times.head()

,user_id,user_session,first_view_time
0,306441847,47641f8a-3aba-471a-8d07-014deccec567,2019-10-01 01:32:09+00:00
1,348609428,bc4879bf-26fb-43bb-8c2b-260d93185a98,2019-10-01 05:10:42+00:00
2,351866718,f791b47c-33f6-470b-8251-08fcddd5a266,2019-10-01 04:36:25+00:00
3,362699320,6c48ec39-b4be-4e62-a646-6c5d92de098c,2019-10-01 03:05:49+00:00
4,370076704,8664f1b9-1405-4d14-a2a8-c15f299cf7e4,2019-10-01 04:03:21+00:00


In [56]:
purchase_times = (
    oct_2019[oct_2019['event_type'] == 'purchase'].groupby(['user_id', 'user_session'])['event_time']
    .min()   # take the min as first view time
    .reset_index()
    .rename(columns = {'event_time' : "first_purchase_time"})
)

time_to_convert = pd.merge(
    view_times, purchase_times, on = ['user_id', 'user_session'], how = 'inner'  # only considering sessions which had purchase
)

time_to_convert.head()

,user_id,user_session,first_view_time,first_purchase_time
0,457360398,b7360dbb-427a-428b-a4fb-1f8872884d4d,2019-10-01 04:05:12+00:00,2019-10-01 04:22:53+00:00
1,504429960,54243b1c-14aa-4c29-9ddd-6607865700a1,2019-10-01 03:41:28+00:00,2019-10-01 04:11:53+00:00
2,512365496,d5aa25d2-afb4-4644-8edc-d8004c03091d,2019-10-01 03:57:17+00:00,2019-10-01 03:57:58+00:00
3,512371939,7737b8d9-9359-420e-8c91-2fa5b1c89279,2019-10-01 04:32:12+00:00,2019-10-01 04:34:23+00:00
4,512373792,8b6814cb-7c0e-4cc9-8224-3a75672affff,2019-10-01 02:45:13+00:00,2019-10-01 02:53:14+00:00


In [57]:
time_to_convert.dtypes
# after the merge, time format became object again, so we need to convert to datetime again

user_id                 int64
user_session           object
first_view_time        object
first_purchase_time    object
dtype: object

In [58]:
# convert to datetime again
time_to_convert['first_view_time'] = pd.to_datetime(
    time_to_convert['first_view_time'], utc=True
)

time_to_convert['first_purchase_time'] = pd.to_datetime(
    time_to_convert['first_purchase_time'], utc=True
)

# calculate the time to convert in minutes
time_to_convert['time_to_purchase_min'] = (
    time_to_convert['first_purchase_time'] - time_to_convert['first_view_time']
).dt.total_seconds()/60

time_to_convert['time_to_purchase_min'] = time_to_convert['time_to_purchase_min'].round(2)

time_to_convert.head()

,user_id,user_session,first_view_time,first_purchase_time,time_to_purchase_min
0,457360398,b7360dbb-427a-428b-a4fb-1f8872884d4d,2019-10-01 04:05:12+00:00,2019-10-01 04:22:53+00:00,17.68
1,504429960,54243b1c-14aa-4c29-9ddd-6607865700a1,2019-10-01 03:41:28+00:00,2019-10-01 04:11:53+00:00,30.42
2,512365496,d5aa25d2-afb4-4644-8edc-d8004c03091d,2019-10-01 03:57:17+00:00,2019-10-01 03:57:58+00:00,0.68
3,512371939,7737b8d9-9359-420e-8c91-2fa5b1c89279,2019-10-01 04:32:12+00:00,2019-10-01 04:34:23+00:00,2.18
4,512373792,8b6814cb-7c0e-4cc9-8224-3a75672affff,2019-10-01 02:45:13+00:00,2019-10-01 02:53:14+00:00,8.02
